# MPA-MLF Room Occupancy — Colab GPU pipeline

**Objectif : top 1 leaderboard**. Submission actuelle = 0.97565 (#4). Top 1 = 0.97819.

## Setup (1 fois)

1. **Runtime → Change runtime type → GPU (T4 ou A100)**.
2. Upload `colab_bundle.zip` via le panneau Files (icône dossier à gauche) OU via Drive.
3. Runs les cellules dans l'ordre. La première cellule dézippe tout.

In [ ]:
# === 0. Dézip + check GPU ===
import os, subprocess
if not os.path.exists('/content/mpa'):
    # attend que colab_bundle.zip soit à la racine ; adapte le path si besoin
    !mkdir -p /content/mpa && cd /content/mpa && unzip -q /content/colab_bundle.zip
!nvidia-smi | head -20
!ls /content/mpa

In [ ]:
# === 1. Variables ===
ROOT = '/content/mpa'
DATA = f'{ROOT}/data'
CSV = f'{ROOT}/csv'
SRC = f'{ROOT}/src'
OUT = f'{ROOT}/outputs'
EXIST = f'{ROOT}/existing'

import os
os.makedirs(OUT, exist_ok=True)
print('train:', f'{DATA}/train_X.npy')
print('test :', f'{DATA}/test_X.npy')
!ls $EXIST

## Étape 1 — Build `stack_test` (pour le pseudo-labeling)

On utilise les 5 CNN seeds CV + features engineered déjà calculés localement (dans `existing/`). Stacking 10-fold LogReg → donne `stack_test.npy` (probas de test) qui servira de source de pseudo-labels.

In [ ]:
!python $SRC/make_stack_test.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --cnn-dir $EXIST/run_strong \
  --cnn-dir $EXIST/run_strong_v2 \
  --features-dir $EXIST/run_features_v2 \
  --output $OUT/stack_test.npy \
  --C 1.0

## Étape 2 — Full-fit (100% train, plein de seeds)

Sur T4 ça devrait tourner 5-10× plus vite que MPS → on se permet **8 seeds** (vs 3 en local). Chaque seed ≈ 2-3 min sur T4 (35 epochs).

In [ ]:
# base=48, 8 seeds
!python $SRC/train_full_fit.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy \
  --test-npy $DATA/test_X.npy \
  --output-dir $OUT/run_full_fit_b48 \
  --seeds 42 1234 7 555 999 2024 31337 101 \
  --epochs 35 --batch 256 --base 48

In [ ]:
# base=64 (plus large), 5 seeds
!python $SRC/train_full_fit.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy \
  --test-npy $DATA/test_X.npy \
  --output-dir $OUT/run_full_fit_b64 \
  --seeds 42 1234 7 555 999 \
  --epochs 35 --batch 256 --base 64

## Étape 3 — Pseudo-labeling (thr=0.99)

Réentraîne sur train + test à haute confiance.

In [ ]:
!python $SRC/train_pseudo.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy \
  --test-npy $DATA/test_X.npy \
  --stack-test-probs $OUT/stack_test.npy \
  --output-dir $OUT/run_pseudo_b48 \
  --seeds 42 1234 7 555 999 \
  --epochs 30 --batch 256 --base 48 --threshold 0.99

In [ ]:
!python $SRC/train_pseudo.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --train-npy $DATA/train_X.npy \
  --test-npy $DATA/test_X.npy \
  --stack-test-probs $OUT/stack_test.npy \
  --output-dir $OUT/run_pseudo_b64 \
  --seeds 42 1234 7 555 999 \
  --epochs 30 --batch 256 --base 64 --threshold 0.99

## Étape 4 — Blend final

Blend CNN seeds + features via stacking + weighted. Génère 5-6 submissions.

In [ ]:
!python $SRC/blend_all.py \
  --train-csv $CSV/y_train_v2.csv \
  --sample-submission $CSV/y_test_submission_example_v2.csv \
  --cnn-dir $EXIST/run_strong \
  --cnn-dir $EXIST/run_strong_v2 \
  --features-dir $EXIST/run_features_v2 \
  --full-test $OUT/run_full_fit_b48/test_full_probs.npy \
  --full-test $OUT/run_full_fit_b64/test_full_probs.npy \
  --full-test $OUT/run_pseudo_b48/test_pseudo_probs.npy \
  --full-test $OUT/run_pseudo_b64/test_pseudo_probs.npy \
  --output-dir $OUT/blend_final

## Étape 5 — Submissions supplémentaires (mix manuel)

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

OUT = Path('/content/mpa/outputs')
test_df = pd.read_csv('/content/mpa/csv/y_test_submission_example_v2.csv').sort_values('id').reset_index(drop=True)

full48 = np.load(OUT / 'run_full_fit_b48/test_full_probs.npy')
full64 = np.load(OUT / 'run_full_fit_b64/test_full_probs.npy')
ps48   = np.load(OUT / 'run_pseudo_b48/test_pseudo_probs.npy')
ps64   = np.load(OUT / 'run_pseudo_b64/test_pseudo_probs.npy')
stack  = np.load(OUT / 'stack_test.npy')

variants = {
    'all_equal':   (full48 + full64 + ps48 + ps64) / 4,
    'pseudo_heavy': 0.25*full48 + 0.25*full64 + 0.25*ps48 + 0.25*ps64,
    'pseudo_only':  0.5*ps48 + 0.5*ps64,
    'full_only':    0.5*full48 + 0.5*full64,
    'stack_pseudo': 0.5*stack + 0.25*ps48 + 0.25*ps64,
    'stack_full':   0.5*stack + 0.25*full48 + 0.25*full64,
    'mega':         0.2*stack + 0.2*full48 + 0.2*full64 + 0.2*ps48 + 0.2*ps64,
    'geom_mean':    np.exp((np.log(full48+1e-9)+np.log(full64+1e-9)+np.log(ps48+1e-9)+np.log(ps64+1e-9))/4),
}

out_sub = OUT / 'submissions'; out_sub.mkdir(exist_ok=True)
for tag, p in variants.items():
    sub = test_df.copy()
    sub['target'] = p.argmax(1).astype(int)
    path = out_sub / f'submission_{tag}.csv'
    sub.to_csv(path, index=False)
    print(tag, path)

## Étape 6 — Télécharge les submissions

In [ ]:
!cd /content/mpa/outputs && zip -r /content/submissions.zip blend_final submissions run_full_fit_b48/test_full_probs.npy run_full_fit_b64/test_full_probs.npy run_pseudo_b48/test_pseudo_probs.npy run_pseudo_b64/test_pseudo_probs.npy stack_test.npy
from google.colab import files
files.download('/content/submissions.zip')

## Tips

- Si Colab te donne un A100 : mets `--batch 512` partout, c'est encore plus rapide.
- Si session disconnect : les outputs sont déjà sur disque Colab, remount Drive pour backup.
- Ordre de priorité des submissions à tester : `blend_final/submission_with_fullfit.csv` → `submissions/submission_pseudo_heavy.csv` → `submissions/submission_stack_pseudo.csv`.